In [ ]:
"""
Network preprocessing (calculate OD)

GPU acceleration via RAPIDS (cudf, cugraph) is expected.
"""

In [1]:
# ============================================================
# 0. Python version & package installation (optional for notebooks)
# ============================================================
# Python version: 3.12.9 | packaged by conda‑forge | (main, Feb 14 2025) [GCC 13.3.0]
import sys, subprocess

import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm


import cudf
import cugraph
os.chdir(os.getcwd())

DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "cities"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "groupednodes"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "groupedlinks"), exist_ok=True)

In [16]:
# ============================================================
# 1. Join population / links to cities
# ------------------------------------------------------------

# Load raw spatial & population files
links = gpd.read_file('raw data/2022-TM-KA-MR-LLV2 도로망(2021년 기준)/02. 링크/ad0022_2021_KA.shp')
nodes = gpd.read_file('raw data/2022-TM-KA-MR-LLV2 도로망(2021년 기준)/01. 노드/ad0102_2021_KA.shp')
cities = gpd.read_file('raw data/bnd_sigungu_00_2023_2Q.shp')
jgg_pop = pd.read_csv('raw data/2023년기준_2022년_인구총괄(총인구).txt', sep='^', header=None)

# Convert coordinates to EPSG:5179
links = links.to_crs(epsg=5179)
nodes = nodes.to_crs(epsg=5179)
cities = cities.to_crs(epsg=5179)

# Pre‑process JGG population table
jgg_pop.columns = ['year', 'jgg_id', 'pop_type', 'pop']
jgg_pop = jgg_pop[jgg_pop['pop_type'] == 'to_in_001']
jgg_pop['SIGUNGU_CD'] = jgg_pop['jgg_id'].apply(lambda x: str(x)[:5])

# Merge the aggregated population data with the cities GeoDataFrame
city_population = jgg_pop.groupby('SIGUNGU_CD')['pop'].sum().reset_index()
cities = cities.merge(city_population, left_on='SIGUNGU_CD', right_on='SIGUNGU_CD', how='left')

# Select midsize cities
midsize_cities = cities[(cities['pop'] >= 50000) & (cities['pop'] < 1000000)]
midsize_cities = midsize_cities[~midsize_cities['SIGUNGU_NM'].str.endswith(('구', '군'))]

# Map Korean city names to English
korean_to_english = {
    "동해시": "Donghae",
    "평택시": "Pyeongtaek",
    "밀양시": "Miryang",
    "순천시": "Suncheon",
    "의왕시": "Uiwang",
    "경산시": "Gyeongsan",
    "목포시": "Mokpo",
    "의정부시": "Uijeongbu",
    "당진시": "Dangjin",
    "구미시": "Gumi",
    "여수시": "Yeosu",
    "거제시": "Geoje",
    "이천시": "Icheon",
    "광양시": "Gwangyang",
    "문경시": "Mungyeong",
    "보령시": "Boryeong",
    "군산시": "Gunsan",
    "진주시": "Jinju",
    "광주시": "Gwangju",
    "오산시": "Osan",
    "군포시": "Gunpo",
    "서귀포시": "Seogwipo",
    "제주시": "Jeju",
    "김해시": "Gimhae",
    "양주시": "Yangju",
    "김천시": "Gimcheon",
    "동두천시": "Dongducheon",
    "제천시": "Jecheon",
    "과천시": "Gwacheon",
    "익산시": "Iksan",
    "논산시": "Nonsan",
    "서산시": "Seosan",
    "김제시": "Gimje",
    "영천시": "Yeongcheon",
    "경주시": "Gyeongju",
    "사천시": "Sacheon",
    "영주시": "Yeongju",
    "하남시": "Hanam",
    "세종시": "Sejong",
    "화성시": "Hwaseong",
    "시흥시": "Siheung",
    "상주시": "Sangju",
    "통영시": "Tongyeong",
    "여주시": "Yeoju",
    "나주시": "Naju",
    "김포시": "Gimpo",
    "광명시": "Gwangmyeong",
    "파주시": "Paju",
    "춘천시": "Chuncheon",
    "부천시": "Bucheon",
    "남원시": "Namwon",
    "남양주시": "Namyangju",
    "속초시": "Sokcho",
    "포천시": "Pocheon",
    "양산시": "Yangsan",
    "충주시": "Chungju",
    "정읍시": "Jeongeup",
    "구리시": "Guri",
    "강릉시": "Gangneung",
    "안성시": "Anseong",
    "원주시": "Wonju",
    "공주시": "Gongju",
    "안동시": "Andong",
    "삼척시": "Samcheok",
    "아산시": "Asan"
}

# Add English name column & save
midsize_cities["SIGUNGU_NM_EN"] = midsize_cities["SIGUNGU_NM"].map(korean_to_english)

# (≈20 s)


In [ ]:
# ============================================================
# 2. Group links & nodes by city & filter by LCC ratio
# ------------------------------------------------------------

# Spatial join and calculate link length
links_with_cities = gpd.sjoin(
    links, 
    midsize_cities[['SIGUNGU_NM', 'SIGUNGU_CD', 'geometry']], 
    how="left", predicate='within'
)
nodes_with_cities = gpd.sjoin(
    nodes, 
    midsize_cities[['SIGUNGU_NM', 'SIGUNGU_CD', 'geometry']], 
    how="left", predicate='within'
)

links_with_cities = links_with_cities[links_with_cities['SIGUNGU_CD'].notnull()]
nodes_with_cities = nodes_with_cities[nodes_with_cities['SIGUNGU_CD'].notnull()]

links_with_cities["LENGTH"] = links_with_cities.geometry.length

# Lists to store results
valid_links = []
valid_nodes = []

# Build city-level networks and check LCC ratio
for city_name, group in tqdm(links_with_cities.groupby('SIGUNGU_NM'), desc='filtering by LCC ratio'):
    G_cugraph = cugraph.Graph()
    edges_cudf = cudf.DataFrame.from_pandas(
        group[['UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'LINK_ID', 'SIGUNGU_NM']]
    )
    
    G_cugraph.from_cudf_edgelist(
        edges_cudf, source='UP_FROM_NO', destination='UP_TO_NODE', edge_attr='LENGTH'
    )
    
    nodes_in_G = edges_cudf[['UP_FROM_NO', 'UP_TO_NODE']].melt(value_name='NODE_ID')['NODE_ID'].unique()

    # Connected components
    cc_df = cugraph.connected_components(G_cugraph)
    cc_size = cc_df.groupby('labels').size().reset_index(name='size')  # ✅ labels로 수정
    largest_cc = cc_size['size'].max()
    total_nodes = len(cc_df)
    lcc_ratio = largest_cc / total_nodes if total_nodes > 0 else 0

    if lcc_ratio >= 0.95:
        valid_links.append(group)

        # Filter nodes that exist in the city graph
        city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name]
        city_nodes = city_nodes[city_nodes['NODE_ID'].isin(nodes_in_G.to_pandas())]
        valid_nodes.append(city_nodes)
    else:
        print(f"[DELETE] {city_name}: total={total_nodes}, largest={largest_cc}, ratio={lcc_ratio:.3f}")


# Merge results
links_with_cities = pd.concat(valid_links, ignore_index=True)
nodes_with_cities = pd.concat(valid_nodes, ignore_index=True)

# Save output
links_with_cities.to_file("data/links_with_cities.gpkg", driver="GPKG")
nodes_with_cities.to_file("data/nodes_with_cities.gpkg", driver="GPKG")

midsize_cities = midsize_cities[midsize_cities['SIGUNGU_NM'].isin(links_with_cities['SIGUNGU_NM'].unique())]
midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG', index=False)

In [ ]:
# ============================================================
# 3. Calculate OD matrices (cuGraph SSSP)
# ------------------------------------------------------------
# 3.1 Ensure output directory exists
if not os.path.exists('data/groupedod/'):
    os.makedirs('data/groupedod/')

processing_city = 0

# Loop over each city
for city_name, group in links_with_cities.groupby('SIGUNGU_NM'):
    start_time = time.time()
    processing_city += 1

    # 3.Skip if OD already computed
    if os.path.exists(f'data/groupedod/{city_name}_od.csv'):
        print(f'{processing_city}/{len(midsize_cities)} {city_name} already processed, skipping')
        continue
    
    # Build city graph in cuGraph
    G_cugraph = cugraph.Graph()
    edges_cudf = cudf.DataFrame.from_pandas(group[['UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'LINK_ID']])
    G_cugraph.from_cudf_edgelist(edges_cudf, source='UP_FROM_NO', destination='UP_TO_NODE', edge_attr='LENGTH')
    
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name]
    city_nodes_list = city_nodes['NODE_ID'].to_list()

    
    # Compute all‑pairs shortest paths (SSSP per source)
    od_frames = []
    def shortest_path(src):
        return cugraph.sssp(G_cugraph, source=src, edge_attr='LENGTH')

    for src in tqdm(city_nodes_list, desc=f'Processing {city_name} {processing_city}/{len(midsize_cities)}'):
        od = shortest_path(src)
        frame = pd.DataFrame({'origin': src,
                              'destination': od['vertex'].to_numpy(),
                              'distance': od['distance'].to_numpy(),
                              'predecessor': od['predecessor'].to_numpy()})
        od_frames.append(frame)
    
    # Concatenate results & post‑process distances
    od_df = pd.concat(od_frames, ignore_index=True)
    od_df = od_df[od_df['origin'] != od_df['destination']]
    od_df['origin'] = od_df['origin'].astype(str)
    od_df['destination'] = od_df['destination'].astype(str)

    node_xy = city_nodes.set_index('NODE_ID')[['X', 'Y']]
    od_df['euclidean_distance'] = np.linalg.norm(node_xy.loc[od_df['origin']].values - node_xy.loc[od_df['destination']].values, axis=1)

    od_df['distance_ratio'] = od_df['distance'] / od_df['euclidean_distance']
    
    
    
    od_df['interval'] = (od_df['euclidean_distance'] // 100).round().astype(str)
    reached = od_df[od_df['predecessor'] != '-1']
    interval_mean = reached.groupby('interval')['distance_ratio'].mean()
    od_df = od_df.merge(interval_mean.rename('mean_ratio'), on='interval', how='left')
    od_df.loc[od_df['predecessor'] == '-1', 'distance'] = od_df['euclidean_distance'] * od_df['mean_ratio']
    od_df['distance_ratio'] = od_df['distance'] / od_df['euclidean_distance']
    od_df.drop(columns=['interval', 'mean_ratio'], inplace=True)

    # Save OD matrix
    od_df.to_csv(f'data/groupedod/{city_name}_od.csv', index=False)
    print(f'{processing_city}/{len(midsize_cities)} {city_name} processed in {time.time() - start_time:.1f} s')

4. Calculate centrality metrics

In [29]:
import pandas as pd
import numpy as np
import os
import glob
from collections import defaultdict
from scipy.stats import linregress
from tqdm import tqdm

bin_size = 100  # in meters (1000 = 1km)
chunk_size = 10_000_000  # adjust based on available memory

# Accumulate bin statistics only
bin_accum = defaultdict(lambda: {'sum_net': 0.0, 'sum_euc': 0.0, 'count': 0})

od_files_path = sorted(glob.glob('data/groupedod/*_od.csv'))
print(f"Total {len(od_files_path)} cities to process")

# City-level progress bar
city_pbar = tqdm(od_files_path, desc='Cities', position=0)

for f in city_pbar:
    city_name = os.path.basename(f).replace('_od.csv', '')
    city_pbar.set_postfix_str(city_name)
    
    # File size for reference (optional)
    file_size_mb = os.path.getsize(f) / (1024**2)
    
    # Read in chunks
    reader = pd.read_csv(
        f,
        usecols=['distance', 'distance_ratio'],
        chunksize=chunk_size
    )
    
    # Chunk-level progress bar
    chunk_pbar = tqdm(reader, desc=f'  {city_name} chunks',
                      position=1, leave=False,
                      unit='chunk')
    
    rows_processed = 0
    for chunk in chunk_pbar:
        # Keep valid rows only
        chunk = chunk[chunk['distance_ratio'].notna() & (chunk['distance'] > 0)]
        if len(chunk) == 0:
            continue
        
        rows_processed += len(chunk)
        chunk_pbar.set_postfix_str(f'{rows_processed:,} rows')
        
        # Recover Euclidean distance
        euclidean = chunk['distance'].values / chunk['distance_ratio'].values
        network = chunk['distance'].values
        
        # Assign bins
        bin_ids = (euclidean // bin_size).astype(int) * bin_size + bin_size / 2
        
        # Accumulate per bin (vectorized)
        df_tmp = pd.DataFrame({
            'bin': bin_ids,
            'net': network,
            'euc': euclidean
        })
        agg = df_tmp.groupby('bin').agg(
            sum_net=('net', 'sum'),
            sum_euc=('euc', 'sum'),
            count=('net', 'size')
        )
        
        for b, row in agg.iterrows():
            bin_accum[b]['sum_net'] += row['sum_net']
            bin_accum[b]['sum_euc'] += row['sum_euc']
            bin_accum[b]['count'] += row['count']
        
        del chunk, df_tmp, agg
    
    chunk_pbar.close()

city_pbar.close()

print(f"\nTotal accumulated bins: {len(bin_accum):,}")

# Compute per-bin average circuity
print("\n[Step 2/3] Aggregating bin statistics...")
bin_stats = pd.DataFrame([
    {'dist_bin': b, 
     'C_bin': v['sum_net'] / v['sum_euc'], 
     'count': v['count']}
    for b, v in tqdm(bin_accum.items(), desc='Aggregating bins')
]).sort_values('dist_bin').reset_index(drop=True)

print(f"Total bins: {len(bin_stats):,}")

# Double-log regression
print("\n[Step 3/3] Double-log regression...")
bin_stats['ln_d'] = np.log(bin_stats['dist_bin'])
bin_stats['ln_C'] = np.log(bin_stats['C_bin'])

slope, intercept, r_value, p_value, std_err = linregress(
    bin_stats['ln_d'], bin_stats['ln_C']
)
zeta_hat = slope
beta_hat = intercept

print(f"\n=== Giacomin & Levinson global fit ===")
print(f"zeta      = {zeta_hat:.5f}")
print(f"beta      = {beta_hat:.5f}")
print(f"r^2       = {r_value**2:.4f}")
print(f"p-value   = {p_value:.2e}")
print(f"std_err   = {std_err:.5f}")
print(f"# of bins = {len(bin_stats)}")
print(f"\nExpected circuity formula: C(d) = exp({beta_hat:.4f}) * d^({zeta_hat:.4f})")

# Save results
bin_stats.to_csv('data/distance_decay_bins.csv', index=False)
with open('data/distance_decay_params.txt', 'w') as fp:
    fp.write(f"zeta={zeta_hat}\n")
    fp.write(f"beta={beta_hat}\n")
    fp.write(f"r_squared={r_value**2}\n")
    fp.write(f"p_value={p_value}\n")
    fp.write(f"n_bins={len(bin_stats)}\n")

print("\nResults saved: data/distance_decay_bins.csv, data/distance_decay_params.txt")


Total 60 cities to process


Cities: 100%|██████████| 60/60 [03:49<00:00,  3.82s/it, 화성시]  



Total accumulated bins: 744

[Step 2/3] Aggregating bin statistics...


Aggregating bins: 100%|██████████| 744/744 [00:00<00:00, 1352063.33it/s]

Total bins: 744

[Step 3/3] Double-log regression...

=== Giacomin & Levinson global fit ===
zeta      = -0.07307
beta      = 0.92134
r^2       = 0.8562
p-value   = 1.10e-314
std_err   = 0.00110
# of bins = 744

Expected circuity formula: C(d) = exp(0.9213) * d^(-0.0731)

Results saved: data/distance_decay_bins.csv, data/distance_decay_params.txt


In [ ]:
# ============================================================
# 4. Calculate centrality metrics
# ------------------------------------------------------------

# Load distance-decay parameters
with open('data/distance_decay_params.txt', 'r') as fp:
    params = dict(line.strip().split('=') for line in fp if '=' in line)
zeta_hat = float(params['zeta'])
beta_hat = float(params['beta'])
print(f"Loaded distance-decay params: zeta={zeta_hat:.5f}, beta={beta_hat:.5f}")
print(f"Expected circuity: C(d) = exp({beta_hat:.4f}) * d^({zeta_hat:.4f})\n")

# Ensure output directories exist
if not os.path.exists('data/groupednodes/'):
    os.makedirs('data/groupednodes/')
if not os.path.exists('data/groupedlinks/'):
    os.makedirs('data/groupedlinks/')

# 4.2 Helper: reconstruct route from predecessor dictionary
def find_route(origin, destination, predecessors):
    route = [destination]
    while (origin, route[-1]) in predecessors:
        route.append(predecessors[(origin, route[-1])])
    return route[::-1]

# Iterate over cities and compute node/link metrics
od_files = nodes_with_cities.groupby('SIGUNGU_NM').count().sort_values('NODE_ID', ascending=True).reset_index()['SIGUNGU_NM'].tolist()
processing_city = 0

for city_name in od_files:
    start_time = time.time()
    processing_city += 1

    node_out = f'data/groupednodes/{city_name}_node_centrality.gpkg'
    link_out = f'data/groupedlinks/{city_name}_link_betweenness.gpkg'
    if os.path.exists(node_out) and os.path.exists(link_out):
        print(f'{processing_city}/{len(od_files)} {city_name} already processed, skipping')
        continue

    print(f'\nprocessing {city_name}: {processing_city}/{len(od_files)}')

    # 4.3.1 Load OD matrix for the city
    od_df = pd.read_csv(f'data/groupedod/{city_name}_od.csv')
    od_df['origin'] = od_df['origin'].astype(str)
    od_df['destination'] = od_df['destination'].astype(str)
    od_df['predecessor'] = od_df['predecessor'].astype(str)

    # --- Compute normalized detour rate per OD pair ---
    mask = (
        od_df['distance_ratio'].notna()
        & (od_df['distance'] > 0)
        & od_df['euclidean_distance'].notna()
        & (od_df['euclidean_distance'] > 0)
    )
    od_df['distance_ratio_normalized'] = np.nan
    if mask.any():
        ratio = od_df.loc[mask, 'distance_ratio'].values
        euclidean = od_df.loc[mask, 'euclidean_distance'].values
        C_expected = np.exp(beta_hat) * (euclidean ** zeta_hat)
        od_df.loc[mask, 'distance_ratio_normalized'] = ratio / C_expected
    # --------------------------------------------------

    od_df_reached = od_df[od_df['predecessor'] != '-1']
    predecessors = od_df_reached.set_index(['origin', 'destination'])['predecessor'].to_dict()

    node_spi_counter = {'points': Counter()}
    nodes_counter = {'points': Counter()}
    edges_counter = {'points': Counter()}

    # Reconstruct routes in chunks to save memory
    chunk_size = 10_000_000
    num_chunks = len(od_df_reached) // chunk_size + 1
    for chunk in range(num_chunks):
        start_idx = chunk * chunk_size
        end_idx = min((chunk + 1) * chunk_size, len(od_df_reached))
        routes_cache = []
        for _, row in tqdm(od_df_reached.iloc[start_idx:end_idx].iterrows(),
                           desc=f'  Processing Chunk {chunk+1}/{num_chunks}', total=end_idx - start_idx):
            route = find_route(row['origin'], row['destination'], predecessors)
            routes_cache.append(route)
        for route in tqdm(routes_cache, desc='     node counting', total=len(routes_cache)):
            node_spi_counter['points'][route[0]] += (len(route)*(len(route)-1))/2
            for node in route:
                nodes_counter['points'][node] += 1
            for i in range(len(route)-1):
                edge = (route[i], route[i+1])
                edges_counter['points'][edge] += 1
        del routes_cache

    # Merge counters back to GeoDataFrames
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name].copy()
    node_route_df = pd.DataFrame(nodes_counter['points'].items(), columns=['NODE_ID', 'betweenness'])
    node_spi_df = pd.DataFrame(node_spi_counter['points'].items(), columns=['NODE_ID', 'spi'])
    city_nodes = city_nodes.merge(node_route_df, on='NODE_ID', how='left')
    city_nodes = city_nodes.merge(node_spi_df, on='NODE_ID', how='left')

    city_links = links_with_cities[links_with_cities['SIGUNGU_NM'] == city_name].copy()
    edge_route_df = pd.DataFrame(edges_counter['points'].items(), columns=['edge', 'betweenness'])
    edge_route_df[['UP_FROM_NO', 'UP_TO_NODE']] = pd.DataFrame(edge_route_df['edge'].tolist(), index=edge_route_df.index)
    edge_route_df = edge_route_df.drop(columns=['edge'])
    edge_route_df['betweenness'] *= 2  # undirected graph adjustment
    city_links = city_links.merge(edge_route_df, on=['UP_FROM_NO', 'UP_TO_NODE'], how='left')

    # Closeness, straightness, and normalized straightness
    centrality_index = od_df.groupby('origin')[
        ['distance', 'distance_ratio', 'distance_ratio_normalized']
    ].mean().reset_index()
    centrality_index.columns = ['NODE_ID', 'closeness', 'straightness', 'straightness_normalized']

    nodes_per_m2 = len(city_nodes) / city_links['LENGTH'].sum()
    od_df['pairs'] = ((od_df['distance']*nodes_per_m2)*((od_df['distance']*nodes_per_m2)-1))/2
    centrality_index['expectedpairs'] = od_df.groupby('origin')['pairs'].sum().reset_index()['pairs']

    city_nodes = city_nodes.merge(centrality_index, on='NODE_ID', how='left')

    # Select final columns and save
    city_nodes = city_nodes[['NODE_ID', 'SIGUNGU_NM', 'geometry', 'betweenness', 'spi', 'expectedpairs',
                             'closeness', 'straightness', 'straightness_normalized']]
    city_links = city_links[['LINK_ID', 'UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'geometry',
                             'SIGUNGU_NM', 'SIGUNGU_CD', 'betweenness']]

    city_nodes.to_file(node_out, driver='GPKG', index=False)
    city_links.to_file(link_out, driver='GPKG', index=False)

    print(f'{processing_city}/{len(od_files)} {city_name} processed in {time.time() - start_time:.1f} s')

In [7]:
import pandas as pd
import numpy as np
import os
import glob
import geopandas as gpd
from tqdm import tqdm

# Load distance-decay parameters
with open('data/distance_decay_params.txt', 'r') as fp:
    params = dict(line.strip().split('=') for line in fp if '=' in line)
zeta_hat = float(params['zeta'])
beta_hat = float(params['beta'])
print(f"Loaded distance-decay params: zeta={zeta_hat:.5f}, beta={beta_hat:.5f}")
print(f"Expected circuity: C(d) = exp({beta_hat:.4f}) * d^({zeta_hat:.4f})\n")

chunk_size = 10_000_000

# Find all node files
node_files = sorted(glob.glob('data/groupednodes/*_node_centrality.gpkg'))
print(f"Total {len(node_files)} city node files to update\n")

city_pbar = tqdm(node_files, desc='Cities', position=0)

for node_file in city_pbar:
    city_name = os.path.basename(node_file).replace('_node_centrality.gpkg', '')
    city_pbar.set_postfix_str(city_name)
    
    od_file = f'data/groupedod/{city_name}_od.csv'
    if not os.path.exists(od_file):
        print(f"  [skip] OD file not found for {city_name}")
        continue
    
    # Load existing node file
    city_nodes = gpd.read_file(node_file)
    
    # Skip if already has the column
    if 'straightness_normalized' in city_nodes.columns:
        city_pbar.write(f"  [skip] {city_name} already has straightness_normalized")
        continue
    
    # Per-node accumulator: {NODE_ID: [sum, count]}
    node_accum = {}
    
    reader = pd.read_csv(
        od_file,
        usecols=['origin', 'distance', 'distance_ratio', 'euclidean_distance'],
        chunksize=chunk_size
    )
    
    chunk_pbar = tqdm(reader, desc=f'  {city_name} chunks',
                      position=1, leave=False, unit='chunk')
    
    rows_processed = 0
    for chunk in chunk_pbar:
        # Valid rows only
        mask = (
            chunk['distance_ratio'].notna()
            & (chunk['distance'] > 0)
            & chunk['euclidean_distance'].notna()
            & (chunk['euclidean_distance'] > 0)
        )
        chunk = chunk[mask]
        if len(chunk) == 0:
            continue
        
        rows_processed += len(chunk)
        chunk_pbar.set_postfix_str(f'{rows_processed:,} rows')
        
        # Vectorized normalized detour rate
        ratio = chunk['distance_ratio'].values
        euclidean = chunk['euclidean_distance'].values
        C_expected = np.exp(beta_hat) * (euclidean ** zeta_hat)
        normalized = ratio / C_expected
        
        # Group by origin and aggregate
        tmp = pd.DataFrame({
            'origin': chunk['origin'].astype(str).values,
            'norm': normalized
        })
        agg = tmp.groupby('origin')['norm'].agg(['sum', 'size'])
        
        for node_id, row in agg.iterrows():
            if node_id in node_accum:
                node_accum[node_id][0] += row['sum']
                node_accum[node_id][1] += row['size']
            else:
                node_accum[node_id] = [row['sum'], row['size']]
        
        del chunk, tmp, agg
    
    chunk_pbar.close()
    
    # Per-node mean
    straightness_norm_df = pd.DataFrame([
        {'NODE_ID': k, 'straightness_normalized': v[0] / v[1]}
        for k, v in node_accum.items()
    ])
    
    # Type alignment for merge
    city_nodes['NODE_ID'] = city_nodes['NODE_ID'].astype(str)
    straightness_norm_df['NODE_ID'] = straightness_norm_df['NODE_ID'].astype(str)
    
    # Merge and save back
    city_nodes = city_nodes.merge(straightness_norm_df, on='NODE_ID', how='left')
    city_nodes.to_file(node_file, driver='GPKG', index=False)
    
    # Print quick summary
    valid_count = city_nodes['straightness_normalized'].notna().sum()
    total_count = len(city_nodes)
    city_pbar.write(
        f"  {city_name}: {valid_count}/{total_count} nodes have normalized straightness "
        f"(mean={city_nodes['straightness_normalized'].mean():.4f})"
    )

city_pbar.close()
print("\nAll city node files updated with straightness_normalized")

Loaded distance-decay params: zeta=-0.07307, beta=0.92134
Expected circuity: C(d) = exp(0.9213) * d^(-0.0731)

Total 60 city node files to update



Cities:   2%|▏         | 1/60 [00:07<07:20,  7.47s/it, 거제시]

  강릉시: 3559/3559 nodes have normalized straightness (mean=0.9812)


Cities:   3%|▎         | 2/60 [00:10<04:55,  5.10s/it, 경산시]   

  거제시: 2405/2405 nodes have normalized straightness (mean=1.0424)


...

  평택시: 5281/5281 nodes have normalized straightness (mean=0.9671)


Cities:  98%|█████████▊| 59/60 [05:23<00:08,  8.11s/it, 화성시]   

  포천시: 2704/2704 nodes have normalized straightness (mean=1.0470)


Cities: 100%|██████████| 60/60 [05:58<00:00,  5.97s/it, 화성시]   

  화성시: 8371/8371 nodes have normalized straightness (mean=1.0086)

All city node files updated with straightness_normalized
